**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**02 | EEG–Gaze Alignment and Paired-Window Quality**

![EEG–gaze timebase, source events and paired-window eligibility](../../images/02_eeg_gaze_alignment_and_quality.svg)

**Research question:** How much time is shared by each selected EEG and eye stream, and which non-overlapping windows contain sufficiently valid eye coordinates for a paired-modality analysis?

**Objective:** Read the fixed source registry from Notebook 01, inspect BIDS-relative timing and source events, and construct an auditable registry of eligible and excluded paired windows.

**Input:** The **six participants and twelve NR/TSR runs** recorded in Notebook 01, with fixed **3 Train, 1 Validation and 2 Test** participant roles. The source EEG and EyeLink files are loaded from the temporary Colab cache or retrieved again when unavailable.

**Window protocol:** **12-second** non-overlapping windows, up to **20 per run**, requiring **at least 60%** finite gaze x/y samples.

**Scientific boundary:** BIDS `StartTime` and nominal sampling frequencies provide a documented timebase mapping, not independently verified inter-device clock drift. This notebook has not yet been run on the selected ZuCo signals.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "-q", repo_url, str(repo_path)], check=True
    )

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eeg_eye_tracking_fusion")

# ---------------------------- 2. Standard Library ----------------------------
import importlib.util
import json
import random
import shutil
import subprocess
import sys
from pathlib import Path

# ---------------------- 3. Colab Dependency Preparation ----------------------
if importlib.util.find_spec("mne") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "mne>=1.6,<2"],
        check=True,
    )

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style

# ----------------- 6. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from zuco_io import open_run, session
from alignment import (
    window_registry, ensure_disjoint_registry, eye_sample_times,
    temporal_overlap,
)

# ---------------------------- 7. Reproducibility -----------------------------
SEED = 17
random.seed(SEED)
np.random.seed(SEED)

# ------------------- 8. Output Directory and Setup Summary -------------------
work = root / "eeg_eye_tracking_fusion"
output_dir = work / "output"
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path("/content/zuco2_source_cache")
raw_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Project outputs:", output_dir)
print("Temporary source cache:", raw_dir)
print("Python:", sys.version.split()[0])
print("Free disk (GiB):", round(
    shutil.disk_usage("/content").free / 2**30, 2,
))


**02. Load the Fixed Source Registry and Alignment Settings**

Read `01_source_manifest.csv`, `01_split_assignments.csv` and `01_protocol.json` from the same recorded experiment. Confirm that each manifest participant has the same split role in the saved split registry.

**Recorded upstream evidence:** six people, twelve source runs and one NR/TSR pair per person. Original source files are cached under `/content/zuco2_source_cache`; if this is a new Colab Runtime, they are retrieved again from ZuCo 2.0 rather than substituted with GazeBase data.


In [ ]:
# =========== 02. Load Fixed Source Records and Alignment Settings ============


required = [
    "01_source_manifest.csv", "01_split_assignments.csv",
    "01_protocol.json",
]
missing = [name for name in required if not (output_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing Notebook 01 source outputs: {missing}"
    )

source = pd.read_csv(
    output_dir / "01_source_manifest.csv", dtype={"participant_id": str}
)
saved_roles = pd.read_csv(
    output_dir / "01_split_assignments.csv", dtype={"participant_id": str}
)
protocol = json.loads((output_dir / "01_protocol.json").read_text())
checked = source[["participant_id", "split"]].merge(
    saved_roles, on="participant_id", how="left",
    suffixes=("_manifest", "_registry"), validate="many_to_one",
)
if checked.split_registry.isna().any() or not checked[
    "split_manifest"
].eq(checked["split_registry"]).all():
    raise ValueError("Source manifest and saved participant roles differ")

duration = float(protocol["window_seconds"])
max_windows = int(protocol["max_windows_per_run"])
min_eye_fraction = float(protocol["min_eye_fraction"])
REQUEST = session()

print("Source participants:", source.participant_id.nunique())
print("Original NR/TSR runs:", len(source))
print("Window duration (s):", duration)
print("Minimum eye-coordinate fraction:", min_eye_fraction)
display(source[["participant_id", "task", "run", "split"]])


**03. Inspect Original EEG, Eye and Event Timing**

Read one original BrainVision EEG stream lazily and load its corresponding compressed EyeLink recording. Inspect EEG sample count and duration, gaze sample count, relative gaze start offset, common time interval and source event types.

**Specific quality question:** The small `sub-YAK / NR` source package has a published EEG duration of approximately **15.2 seconds**. Its usable paired-window count depends on actual common duration and gaze validity; the apparent file size alone is not a reason to exclude it.


In [ ]:
# ============= 03. Inspect Raw Temporal Overlap and Event Onsets =============

run = source.iloc[0]
raw, eye, events, eye_meta, eeg_meta = open_run(run, raw_dir, REQUEST)
t_eye = eye_sample_times(eye_meta, len(eye))
overlap = temporal_overlap(raw.info['sfreq'], raw.n_times,
                           eye_meta, len(eye))
print('Selected run:', run.participant_id, run.task, run.run)
print('EEG samples:', raw.n_times, 'Eye samples:', len(eye))
print('Common seconds:', overlap, 'Eye start:', eye_meta['StartTime'])
print('Recorded event types:', events.trial_type.value_counts().head())
raw.close()


**04. Build an Auditable Paired-Window Registry**

Apply the stored **12-second** window protocol separately to each run. Keep original participant, task, run and split identities alongside gaze-valid fractions and the eligibility decision.

**Scientific check:** A model comparison cannot treat an absent NR or TSR recording as a valid paired source. Report missing or zero-eligible-window participant/task groups explicitly; a source-quality failure must not be silently hidden, reassigned to another participant or ignored when defining the experiment.


In [ ]:
# ============== 04. Build Eligible Paired Windows for Every Run ==============

window_parts, quality = [], []
for _, run in source.iterrows():
    raw, eye, events, eye_meta, _ = open_run(run, raw_dir, REQUEST)
    try:
        table, report = window_registry(
            run.to_dict(), raw, eye, eye_meta, events,
            window_seconds=duration,
            max_windows=max_windows,
            min_eye_fraction=min_eye_fraction)
        window_parts.append(table)
        quality.append(report)
    finally:
        raw.close()
    print(run.participant_id, run.task,
          report['eligible_windows'], 'eligible windows')
registry = pd.concat(window_parts, ignore_index=True)
report_table = pd.DataFrame(quality)

# Report all original run/task groups, including those with no windows.
print("Paired-window eligibility by original run:")
display(report_table[[
    "participant_id", "task", "run", "eeg_seconds",
    "eye_seconds", "total_windows", "eligible_windows",
    "eye_start_seconds",
]])
missing_pairs = report_table.loc[
    report_table.eligible_windows.eq(0),
    ["participant_id", "task", "run", "eeg_seconds", "eye_seconds"],
]
if not missing_pairs.empty:
    print("Runs with no eligible paired windows:")
    display(missing_pairs)
    raise ValueError(
        "At least one selected NR/TSR run has no eligible 12-second "
        "EEG–gaze window. Review the reported source duration and gaze "
        "coverage before proceeding; do not silently change the cohort."
    )

ensure_disjoint_registry(registry)
eligible = registry.loc[registry.eligible].copy()
print("Total source windows:", len(registry))
print("Eligible paired windows:", len(eligible))
print("Independent participants with eligible windows:",
      eligible.participant_id.nunique())


**05. Examine Gaze Coverage and Alignment Limits**

Plot the observed distribution of finite eye-coordinate coverage and the number of eligible windows per task. Highlight the stored minimum valid fraction **0.60**, and report source EEG/eye durations and BIDS timing offsets.

**Interpretation:** The figure is an acquisition-quality diagnostic. Neither this plot nor equal nominal sample rates proves drift-free signal synchronisation. Independent timing markers would be needed for a stronger synchrony claim.


In [ ]:
# ============ 05. Examine Eye Coverage and Source Event Alignment ============

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for task, part in registry.groupby('task'):
    axes[0].hist(part.eye_valid_fraction, bins=np.linspace(0, 1, 16),
                 alpha=0.6, label=task)
axes[0].axvline(min_eye_fraction, ls='--', color='black',
                label='Eligibility threshold')
axes[0].set(title='Eye-coordinate coverage per temporal window',
            xlabel='Finite gaze fraction', ylabel='Windows')
axes[0].legend()
report_table.groupby('task')[['eligible_windows']].mean().plot(
    kind='bar', ax=axes[1], legend=False)
axes[1].set(title='Eligible windows per original run',
            ylabel='Mean eligible windows', xlabel='Task')
fig.tight_layout()
fig.savefig(output_dir / '02_alignment_quality.png', dpi=170)
plt.show()


**06. Save Paired Windows and Source Quality Evidence**

Write `02_window_registry.csv`, `02_alignment_quality.csv`, `02_window_summary.csv` and the `02_alignment_quality.png` figure. Every accepted source window retains its participant, task and split identifiers, and excluded windows remain visible in the registry.

This stage produces aligned-window metadata, **not** EEG features, a trained classifier or a clinical interpretation.


In [ ]:
# ========== 06. Save Window Registry and Source Alignment Evidence ===========

registry.to_csv(output_dir / '02_window_registry.csv', index=False)
report_table.to_csv(output_dir / '02_alignment_quality.csv',
                    index=False)
summary = (registry.groupby(['split', 'task'])
           .agg(total=('window_id', 'size'),
                eligible=('eligible', 'sum'),
                people=('participant_id', 'nunique'))
           .reset_index())
summary.to_csv(output_dir / '02_window_summary.csv', index=False)
display(summary)
print("Window registry rows:", len(registry))
print("Unique source window IDs:", registry.window_id.nunique())
print("Distinct participant partitions:",
      registry.groupby("participant_id")["split"].nunique().max())
print("Saved stage-02 files:",
      ", ".join(sorted(p.name for p in output_dir.glob("02_*"))))


**07. Interpretation and Next Dependency**

Summarise the number of eligible source windows and independent people in each participant partition, the range of run durations and the timing assumptions. If a required NR/TSR run yields no eligible windows, examine that source before proceeding; do not change the cohort or window length solely to increase sample counts.

**Next:** Notebook 03 extracts EEG and eye-specific features from these exact accepted windows. Test participants remain excluded from model fitting and selection.


In [ ]:
# ================== 07. Interpretation and Next Dependency ===================

print('Eligible windows:', int(eligible.shape[0]))
print('Independent people:', eligible.participant_id.nunique())
print('Independent drift checked:', False)
print('Next: 03_preprocessing_features_and_neurophysiology.ipynb')

# ---------------------- 2. Package Stage-Owned Outputs -----------------------
download_outputs_from_colab(
    "eeg_eye_tracking_fusion/output", "02"
)
